In [0]:
%run /Workspace/Users/molugurikoushik68@gmail.com/banking-agent/notebooks/config_setup

In [0]:
"""
Document Processor for Banking Policies
========================================

Reads banking policies and splits into semantic chunks.
Each chunk represents a complete idea (not cut mid-sentence).

Key concepts:
1. Read file: Load banking_policies.txt
2. Clean: Remove extra whitespace, special characters
3. Split sentences: Find natural boundaries (. ! ?)
4. Create chunks: Group sentences into 500-word chunks
5. Overlap: 100 words overlap between chunks for context
"""

import os
from typing import List, Tuple

# ============================================
# 1. FUNCTION: Clean Text
# ============================================

def clean_text(text: str) -> str:
    """
    Clean raw text for processing.
    
    What it does:
    - Remove extra whitespace
    - Remove special characters (but keep sentences)
    - Fix line breaks
    
    Why:
    - Raw text has inconsistent spacing/formatting
    - Cleaning ensures consistent processing
    
    Args:
        text: Raw text from file
        
    Returns:
        Cleaned text
        
    Example:
        dirty = "Account  policies\\n\\n  1. Minimum balance"
        clean = clean_text(dirty)
        # Returns: "Account policies\\n1. Minimum balance"
    """
    # Remove extra spaces (but keep single spaces)
    text = ' '.join(text.split())
    
    # Remove common unnecessary characters but keep periods (sentence boundaries)
    # Keep: . ! ? : - ( ) [ ] " ' (important for banking terms)
    # Remove: extra spaces, newlines
    text = text.replace('\n\n', '. ')
    text = text.replace('\n', ' ')
    
    # Fix spacing around periods (ensure "word." not "word .")
    text = text.replace(' .', '.')
    text = text.replace(' !', '!')
    text = text.replace(' ?', '?')
    
    return text

# ============================================
# 2. FUNCTION: Split Into Sentences
# ============================================

def split_into_sentences(text: str) -> List[str]:
    """
    Split text into sentences.
    
    What it does:
    - Finds sentence boundaries (. ! ?)
    - Returns list of sentences
    
    Why:
    - Sentences are natural meaning units
    - Chunking at sentence boundaries preserves context
    
    Args:
        text: Cleaned text
        
    Returns:
        List of sentences
        
    Example:
        text = "Transfer limit is $10,000. Fee is $2.50."
        result = split_into_sentences(text)
        # Returns: ["Transfer limit is $10,000.", "Fee is $2.50."]
    """
    # Simple approach: split on . ! ?
    # But be careful not to split on "U.S.A." or abbreviations
    
    # Split on sentence-ending punctuation
    sentences = []
    current = ""
    
    for char in text:
        current += char
        # If we hit a sentence-ending punctuation followed by space
        if char in '.!?' and len(current) > 2:
            # Don't split if it looks like an abbreviation (e.g., "U.S.")
            if not (len(current) > 2 and current[-3] == '.'):
                sentences.append(current.strip())
                current = ""
    
    # Add remaining text
    if current.strip():
        sentences.append(current.strip())
    
    return sentences

# ============================================
# 3. FUNCTION: Create Overlapping Chunks
# ============================================

def create_chunks(sentences: List[str], 
                 chunk_size: int = 500, 
                 overlap: int = 100) -> List[str]:
    """
    Create overlapping chunks from sentences.
    
    What it does:
    1. Groups sentences into chunks of ~500 words
    2. Creates overlap so last 100 words of Chunk 1 = first 100 of Chunk 2
    3. Returns list of chunks
    
    Why:
    - 500 words = good size for LLM processing
    - Overlap prevents losing info at chunk boundaries
    
    Args:
        sentences: List of sentences
        chunk_size: Target words per chunk (default 500)
        overlap: Words to overlap between chunks (default 100)
        
    Returns:
        List of text chunks
        
    Example:
        sentences = ["Policy 1.", "Policy 2.", "Policy 3.", ...]
        chunks = create_chunks(sentences, chunk_size=500, overlap=100)
        # Chunk 0: first 500 words
        # Chunk 1: word 400-900 (100 word overlap with Chunk 0)
        # Chunk 2: word 800-1300 (100 word overlap with Chunk 1)
    """
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    for sentence in sentences:
        # Count words in sentence
        sentence_word_count = len(sentence.split())
        
        # If adding this sentence exceeds chunk_size, save chunk and start new
        if current_word_count + sentence_word_count > chunk_size and current_chunk:
            # Join sentences into chunk
            chunk_text = ' '.join(current_chunk)
            chunks.append(chunk_text)
            
            # Start new chunk with overlap
            # Keep last ~100 words for context
            words_in_chunk = chunk_text.split()
            overlap_words = words_in_chunk[-overlap:] if len(words_in_chunk) > overlap else words_in_chunk
            overlap_text = ' '.join(overlap_words)
            
            # Start new chunk with overlap
            current_chunk = [overlap_text] if overlap_text else []
            current_word_count = len(' '.join(current_chunk).split())
        
        # Add sentence to current chunk
        current_chunk.append(sentence)
        current_word_count += sentence_word_count
    
    # Add final chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks

# ============================================
# 4. FUNCTION: Load and Process Policies
# ============================================

def process_policies(file_path: str, 
                    chunk_size: int = 500, 
                    overlap: int = 100) -> Tuple[List[str], dict]:
    """
    Main function: Read policies file and return chunks.
    
    What it does:
    1. Read file from /Workspace/
    2. Clean text
    3. Split into sentences
    4. Create overlapping chunks
    5. Return chunks + metadata
    
    Why:
    - One function does entire pipeline
    - Easier to use and maintain
    
    Args:
        file_path: Path to banking_policies.txt
        chunk_size: Words per chunk (default 500)
        overlap: Overlap words (default 100)
        
    Returns:
        (chunks, metadata) where:
        - chunks: List of text chunks
        - metadata: Dict with info about chunks
        
    Example:
        chunks, meta = process_policies(
            "/Workspace/Users/.../banking_policies.txt",
            chunk_size=500,
            overlap=100
        )
        print(f"Created {meta['num_chunks']} chunks")
        print(f"Chunk 0: {chunks[0][:100]}...")
    """
    print(f"Processing policies from: {file_path}")
    
    # Step 1: Read file
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()
        print(f"✓ Read file: {len(raw_text)} characters")
    except FileNotFoundError:
        print(f"✗ Error: File not found at {file_path}")
        return [], {}
    except Exception as e:
        print(f"✗ Error reading file: {e}")
        return [], {}
    
    # Step 2: Clean text
    print("✓ Cleaning text...")
    clean = clean_text(raw_text)
    print(f"  Original: {len(raw_text)} chars")
    print(f"  Cleaned: {len(clean)} chars")
    
    # Step 3: Split into sentences
    print("✓ Splitting into sentences...")
    sentences = split_into_sentences(clean)
    print(f"  Total sentences: {len(sentences)}")
    print(f"  First sentence: {sentences[0][:80]}...")
    
    # Step 4: Create chunks
    print(f"✓ Creating chunks (size: {chunk_size} words, overlap: {overlap})...")
    chunks = create_chunks(sentences, chunk_size=chunk_size, overlap=overlap)
    
    # Step 5: Create metadata
    total_words = sum(len(chunk.split()) for chunk in chunks)
    metadata = {
        'num_chunks': len(chunks),
        'total_words': total_words,
        'avg_chunk_size': total_words / len(chunks) if chunks else 0,
        'chunk_size_target': chunk_size,
        'overlap': overlap,
        'num_sentences': len(sentences),
    }
    
    print(f"\n" + "="*60)
    print("CHUNKING SUMMARY")
    print("="*60)
    print(f"✓ Created {metadata['num_chunks']} chunks")
    print(f"✓ Total words: {metadata['total_words']}")
    print(f"✓ Avg chunk size: {metadata['avg_chunk_size']:.0f} words")
    print(f"✓ Sentences: {metadata['num_sentences']}")
    print("="*60)
    
    return chunks, metadata

# ============================================
# 5. FUNCTION: Display Chunks
# ============================================

def display_chunks(chunks: List[str], num_to_show: int = 3):
    """
    Display sample chunks for inspection.
    
    Args:
        chunks: List of chunks
        num_to_show: How many chunks to display
    """
    print(f"\nShowing first {min(num_to_show, len(chunks))} chunks:\n")
    
    for i, chunk in enumerate(chunks[:num_to_show]):
        print(f"--- CHUNK {i} ({len(chunk.split())} words) ---")
        print(chunk)
        print()

# ============================================
# 6. MAIN EXECUTION
# ============================================

# Configuration is imported from config_setup notebook (Cell 1)
# Using: POLICIES_FILE, CHUNK_SIZE, CHUNK_OVERLAP

print("="*60)
print("DOCUMENT PROCESSOR: Banking Policies")
print("="*60)

# Process policies
chunks, metadata = process_policies(
    POLICIES_FILE,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
)

# Display samples
if chunks:
    display_chunks(chunks, num_to_show=2)
    
    # Save chunks for next step (embeddings)
    # We'll use them in the embeddings notebook
    print(f"\n✓ {len(chunks)} chunks ready for embedding!")
else:
    print("✗ No chunks created. Check file path.")

# ============================================
# 7. EXPORT CHUNKS (for next step)
# ============================================

# Store in variable for embeddings generator
processed_chunks = chunks
print(f"\n✓ Chunks stored in 'processed_chunks' variable")
print(f"✓ Ready for embeddings generation (next step)")